# Belief Shift Experiment — Centaur Mode

Runs Zhile's framing × refutation experiment using Centaur-70B as the human agent
and llama3.1:70b as the AI persuader.

**GPU requirement**: A100 or H100 required (70B model needs ~40GB+ VRAM). T4 will NOT work.

**Models**:
- AI persuader: `llama3.1:70b` (two-phase execution to avoid running two 70B models simultaneously)
- Human agent: `Centaur-70B` (LoRA adapter on `llama3.1:70b`)

**Two-phase approach**:
1. Phase 1: Generate all persuasive texts using llama3.1:70b, save to CSV
2. Phase 2: Load texts, run Centaur-70B evaluation

**Independent variables**:
- Framing: gain / loss / none (control)
- Refutation: one-sided / two-sided / none (control)

**Conditions per claim**: 2 text types × 3 framings × 3 refutations = 18 experimental + 1 control = 19

**Output**: `centaur_results.csv` and `centaur_generated_texts.csv`

In [ ]:
# Install and start Ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Pull the 8B model for AI persuader (~4.9GB)
!ollama pull llama3.1:8b-instruct-q4_K_M
print("\nOllama + 8B model ready!")

## Setup Centaur-70B

Downloads the LoRA adapter from HuggingFace, converts it to GGUF format using llama.cpp,
pulls the 70B base model, and creates the centaur model in Ollama.

This cell takes a while (~20-30 min for the 70B model pull).

In [ ]:
# Install dependencies for adapter conversion
!pip install -q huggingface_hub gguf safetensors sentencepiece

# Download Centaur LoRA adapter from HuggingFace
from huggingface_hub import snapshot_download
adapter_path = snapshot_download(repo_id="marcelbinz/Llama-3.1-Centaur-70B-adapter")
print(f"Adapter downloaded to: {adapter_path}")

# Clone llama.cpp for GGUF conversion tools
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp

# Convert LoRA adapter to GGUF format
!python /content/llama.cpp/convert_lora_to_gguf.py \
    "{adapter_path}" \
    --outfile /content/centaur-adapter.gguf \
    --outtype f16
print("\nAdapter converted to GGUF!")

# Pull base model (~40GB quantized, needs A100/H100)
!ollama pull llama3.1:70b

# Create centaur model with adapter via Modelfile
with open("/content/Modelfile.centaur", "w") as f:
    f.write("FROM llama3.1:70b\n")
    f.write("ADAPTER /content/centaur-adapter.gguf\n")
!ollama create centaur -f /content/Modelfile.centaur

# Verify model is available
!ollama list
print("\nCentaur model ready!")

In [ ]:
# Clone the experiment repository
!git clone https://github.com/LLY780/beliefshiftexperiment.git
%cd beliefshiftexperiment
!ls *.csv

## Experiment Code

Centaur mode functions from `shift4.py`. Uses temperature=0 for Centaur evaluation
to ensure deterministic initial beliefs per claim.

Includes two-phase functions: `centaur_phase1_generate` (text generation with specified persuader)
and `centaur_phase2_evaluate` (Centaur belief evaluation using pre-generated texts).

In [ ]:
import ollama
import pandas as pd
import time

# Text type definitions (shared with default mode)
texts = {
    "comment": "Generate a comment based on the requirements",
    "paraphrase": "Generate a paraphrase based on the requirements"
}

# Centaur experiment conditions
centaur_framings = {
    "gain": "Frame your response around benefits, positive outcomes, and what can be gained",
    "loss": "Frame your response around risks, negative consequences, and what could be lost",
    "none": "Respond naturally without any specific framing direction"
}

centaur_refutations = {
    "one-sided": "Present only arguments that support your position. Do not mention, acknowledge, or address any counterarguments or opposing views.",
    "two-sided": "First acknowledge the strongest counterargument against your position, then refute it with evidence before presenting your main argument.",
    "none": "Respond naturally without any specific argumentative structure."
}


def centaur_respond(claim, text_type, framing, refutation):
    """Generate AI persuasive text for centaur experiment."""
    return ollama.generate(
        model="llama3.1:8b-instruct-q4_K_M",
        prompt=(
            f"Generate a response to this claim: {claim}. "
            f"Type: {text_type}, {texts[text_type]}. "
            f"Framing: {centaur_framings[framing]}. "
            f"Refutation style: {centaur_refutations[refutation]}. "
            f"Do not include requirements in your response. Provide your response now:"
        ),
        options={"temperature": 0.25, "num_predict": 250}
    )["response"]


def centaur_evaluate(history):
    """Get belief rating from Centaur model (temperature=0 for determinism)."""
    raw = ollama.chat(
        model="centaur",
        messages=history,
        options={"temperature": 0, "num_predict": 1}
    )['message']['content']
    for char in raw.strip():
        if char in "01234":
            return int(char)
    raise ValueError(f"No valid digit in response: '{raw}'")


def centaur_run_trial(claim, text_type, framing, refutation):
    """Run one experimental trial: initial belief -> show AI text -> final belief."""
    history = [{"role": "user",
        "content": (
            f"You are presented with this claim: {claim} "
            "Rate your belief: 0=strongly disagree, 1=disagree, 2=neutral, "
            "3=agree, 4=strongly agree. Respond with ONLY a single digit."
        )}]
    initial_belief = centaur_evaluate(history)
    gen_text = centaur_respond(claim, text_type, framing, refutation)
    history.append({"role": "assistant", "content": str(initial_belief)})
    history.append({"role": "user",
        "content": (
            f"Now consider this AI {text_type}: {gen_text} "
            "Rate your belief again: 0=strongly disagree, 1=disagree, 2=neutral, "
            "3=agree, 4=strongly agree. Respond with ONLY a single digit."
        )})
    final_belief = centaur_evaluate(history)
    return initial_belief, final_belief, final_belief - initial_belief, gen_text


def centaur_run_control(claim):
    """Control trial: ask belief twice with no AI text (shift should be 0 with temp=0)."""
    history = [{"role": "user",
        "content": (
            f"You are presented with this claim: {claim} "
            "Rate your belief: 0=strongly disagree, 1=disagree, 2=neutral, "
            "3=agree, 4=strongly agree. Respond with ONLY a single digit."
        )}]
    initial_belief = centaur_evaluate(history)
    history.append({"role": "assistant", "content": str(initial_belief)})
    history.append({"role": "user",
        "content": (
            f"Rate your belief on the same claim again: {claim} "
            "0=strongly disagree, 1=disagree, 2=neutral, 3=agree, 4=strongly agree. "
            "Respond with ONLY a single digit."
        )})
    final_belief = centaur_evaluate(history)
    return initial_belief, final_belief, final_belief - initial_belief


def centaur_run_all(n=None):
    """Run the full centaur experiment across all claims and conditions."""
    facts_df = pd.read_csv("fact.csv")
    opins_df = pd.read_csv("opinion.csv")
    claims = []
    for _, row in facts_df.iterrows():
        claims.append((row["claim"], "fact"))
    for _, row in opins_df.iterrows():
        claims.append((row["claim"], "opinion"))
    if n is not None:
        claims = claims[:n]

    results = []
    gen_texts = []
    total = len(claims) * 19
    count = 0

    for claim, claim_type in claims:
        print(f"\nClaim ({claim_type}): {claim}")
        # Control condition (no AI text)
        count += 1
        print(f"\t[{count}/{total}] control (no AI text)")
        init, final, shift = centaur_run_control(claim)
        results.append({
            "claim": claim, "claim_type": claim_type,
            "text_type": "none", "framing": "none", "refutation": "none",
            "initial_belief": init, "final_belief": final, "shift": shift
        })
        if shift != 0:
            print(f"\t  WARNING: control shift={shift} (expected 0)")
        # Experimental conditions: 2 text_types x 3 framings x 3 refutations = 18
        for text_type in texts:
            for framing in centaur_framings:
                for refutation in centaur_refutations:
                    count += 1
                    print(f"\t[{count}/{total}] {text_type} | {framing} | {refutation}")
                    init, final, shift, gen_text = centaur_run_trial(
                        claim, text_type, framing, refutation)
                    results.append({
                        "claim": claim, "claim_type": claim_type,
                        "text_type": text_type, "framing": framing,
                        "refutation": refutation,
                        "initial_belief": init, "final_belief": final, "shift": shift
                    })
                    gen_texts.append({
                        "claim": claim, "text_type": text_type,
                        "framing": framing, "refutation": refutation,
                        "generated_text": gen_text
                    })

    pd.DataFrame(results).to_csv("centaur_results.csv", index=False)
    pd.DataFrame(gen_texts).to_csv("centaur_generated_texts.csv", index=False)
    print(f"\nResults saved to centaur_results.csv ({len(results)} rows)")
    print(f"Generated texts saved to centaur_generated_texts.csv ({len(gen_texts)} rows)")




def centaur_respond_with_model(claim, text_type, framing, refutation, model):
    """Generate AI persuasive text using a specified model."""
    return ollama.generate(
        model=model,
        prompt=(
            f"Generate a response to this claim: {claim}. "
            f"Type: {text_type}, {texts[text_type]}. "
            f"Framing: {centaur_framings[framing]}. "
            f"Refutation style: {centaur_refutations[refutation]}. "
            f"Do not include requirements in your response. Provide your response now:"
        ),
        options={"temperature": 0.25, "num_predict": 250}
    )["response"]


def centaur_eval_with_text(claim, text_type, gen_text):
    """Evaluate belief shift using pre-generated AI text."""
    history = [{"role": "user",
        "content": (
            f"You are presented with this claim: {claim} "
            "Rate your belief: 0=strongly disagree, 1=disagree, 2=neutral, "
            "3=agree, 4=strongly agree. Respond with ONLY a single digit."
        )}]
    initial_belief = centaur_evaluate(history)
    history.append({"role": "assistant", "content": str(initial_belief)})
    history.append({"role": "user",
        "content": (
            f"Now consider this AI {text_type}: {gen_text} "
            "Rate your belief again: 0=strongly disagree, 1=disagree, 2=neutral, "
            "3=agree, 4=strongly agree. Respond with ONLY a single digit."
        )})
    final_belief = centaur_evaluate(history)
    return initial_belief, final_belief, final_belief - initial_belief


def centaur_phase1_generate(n=None, persuader_model="llama3.1:8b-instruct-q4_K_M"):
    """Phase 1: Generate all persuasive texts and save to CSV."""
    facts_df = pd.read_csv("fact.csv")
    opins_df = pd.read_csv("opinion.csv")
    claims = []
    for _, row in facts_df.iterrows():
        claims.append((row["claim"], "fact"))
    for _, row in opins_df.iterrows():
        claims.append((row["claim"], "opinion"))
    if n is not None:
        claims = claims[:n]

    gen_texts = []
    total = len(claims) * len(texts) * len(centaur_framings) * len(centaur_refutations)
    count = 0

    for claim, claim_type in claims:
        print(f"\nClaim ({claim_type}): {claim}")
        for text_type in texts:
            for framing in centaur_framings:
                for refutation in centaur_refutations:
                    count += 1
                    print(f"\t[{count}/{total}] {text_type} | {framing} | {refutation}")
                    gen_text = centaur_respond_with_model(
                        claim, text_type, framing, refutation, persuader_model)
                    gen_texts.append({
                        "claim": claim, "text_type": text_type,
                        "framing": framing, "refutation": refutation,
                        "generated_text": gen_text
                    })

    pd.DataFrame(gen_texts).to_csv("centaur_generated_texts.csv", index=False)
    print(f"\nPhase 1 complete: {len(gen_texts)} texts saved to centaur_generated_texts.csv")


def centaur_phase2_evaluate(n=None):
    """Phase 2: Load generated texts and run Centaur evaluation."""
    gen_df = pd.read_csv("centaur_generated_texts.csv")

    facts_df = pd.read_csv("fact.csv")
    opins_df = pd.read_csv("opinion.csv")
    claims = []
    for _, row in facts_df.iterrows():
        claims.append((row["claim"], "fact"))
    for _, row in opins_df.iterrows():
        claims.append((row["claim"], "opinion"))
    if n is not None:
        claims = claims[:n]

    results = []
    total = len(claims) * 19
    count = 0

    for claim, claim_type in claims:
        print(f"\nClaim ({claim_type}): {claim}")
        # Control condition
        count += 1
        print(f"\t[{count}/{total}] control (no AI text)")
        init, final, shift = centaur_run_control(claim)
        results.append({
            "claim": claim, "claim_type": claim_type,
            "text_type": "none", "framing": "none", "refutation": "none",
            "initial_belief": init, "final_belief": final, "shift": shift
        })
        if shift != 0:
            print(f"\t  WARNING: control shift={shift} (expected 0)")
        # Experimental conditions using pre-generated texts
        for text_type in texts:
            for framing in centaur_framings:
                for refutation in centaur_refutations:
                    count += 1
                    print(f"\t[{count}/{total}] {text_type} | {framing} | {refutation}")
                    match = gen_df[
                        (gen_df["claim"] == claim) &
                        (gen_df["text_type"] == text_type) &
                        (gen_df["framing"] == framing) &
                        (gen_df["refutation"] == refutation)
                    ]
                    gen_text = match.iloc[0]["generated_text"]
                    init, final, shift = centaur_eval_with_text(
                        claim, text_type, gen_text)
                    results.append({
                        "claim": claim, "claim_type": claim_type,
                        "text_type": text_type, "framing": framing,
                        "refutation": refutation,
                        "initial_belief": init, "final_belief": final, "shift": shift
                    })

    pd.DataFrame(results).to_csv("centaur_results.csv", index=False)
    print(f"\nPhase 2 complete: {len(results)} results saved to centaur_results.csv")


print("Centaur experiment code loaded!")

## Sanity Check

Run 2 claims through both phases to verify everything works.
Phase 1 generates texts with llama3.1:70b, Phase 2 evaluates with Centaur.
Control conditions should show shift=0.

In [ ]:
s = time.time()

print("=== Phase 1: Generating persuasive texts (2 claims, llama3.1:70b) ===")
centaur_phase1_generate(n=2, persuader_model="llama3.1:70b")

print("\n=== Phase 2: Evaluating with Centaur (2 claims) ===")
centaur_phase2_evaluate(n=2)

print(f"\nSanity check completed in {time.time()-s:.1f}s")

## Run Full Experiment

240 claims × 19 conditions = 4,560 trials.

Phase 1 (text generation) and Phase 2 (evaluation) run serially to avoid loading
two 70B models simultaneously.

In [ ]:
s = time.time()

print("=== Phase 1: Generating persuasive texts (llama3.1:70b) ===")
centaur_phase1_generate(persuader_model="llama3.1:70b")

print("\n=== Phase 2: Evaluating with Centaur ===")
centaur_phase2_evaluate()

print(f"\nExperiment completed in {time.time()-s:.1f}s")

## Download Results

In [ ]:
from google.colab import files
files.download("centaur_results.csv")
files.download("centaur_generated_texts.csv")